In [13]:
# --- IMPORTS ---
import xarray as xr
import numpy as np
import os

print("[INFO] Libraries loaded successfully")

[INFO] Libraries loaded successfully


In [ ]:
# --- CONFIGURATION ---
base_path = "../data/data/"

drought_threshold = -0.84
wet_threshold = 0.84

target_months = [2, 5, 8, 11]

month_names = {
    2: 'February',
    5: 'May',
    8: 'August',
    11: 'November'
}

print("[INFO] Configuration set")
print(f"Drought threshold: {drought_threshold}")
print(f"Wet threshold: {wet_threshold}")

[INFO] Configuration set
Drought threshold: -0.84
Wet threshold: 0.84


In [15]:
# --- LOAD DATA ---
print("[INFO] Loading datasets...")

ds_spi = xr.open_dataset(os.path.join(base_path, "SPI_ERA5_Land_1980_2023_NA.nc"), decode_times=False)
ds_spei = xr.open_dataset(os.path.join(base_path, "SPEI_ERA5_Land_1980_2023_NA.nc"), decode_times=False)
ds_ssmi = xr.open_dataset(os.path.join(base_path, "SSMI_ERA5_Land_1980_2023_NA.nc"), decode_times=False)
ds_mask = xr.open_dataset(os.path.join(base_path, "mask_NA.nc"))

spi = ds_spi["spi"]
spei = ds_spei["spei"]
ssmi = ds_ssmi["ssmi_03"].rename({'latitude': 'lat', 'longitude': 'lon'})
mask = ds_mask["mask"]

print("[INFO] Data loaded")
print("SPI shape:", spi.shape)
print("SPEI shape:", spei.shape)
print("SSMI shape:", ssmi.shape)

[INFO] Loading datasets...
[INFO] Data loaded
SPI shape: (528, 151, 600)
SPEI shape: (528, 149, 600)
SSMI shape: (528, 151, 600)


In [16]:
# --- INTERPOLATION ---
print("[INFO] Interpolating to common grid...")

spei_interp = spei.interp(lat=spi.lat, lon=spi.lon, method='linear')
ssmi_interp = ssmi.interp(lat=spi.lat, lon=spi.lon, method='linear')

# Adjust mask if needed
if not (len(mask.lat) == len(spi.lat) and len(mask.lon) == len(spi.lon)):
    print("[WARNING] Regridding mask...")
    mask = mask.interp(lat=spi.lat, lon=spi.lon, method='nearest')

valid_region = ~np.isnan(mask)

print("[INFO] Interpolation complete")

[INFO] Interpolating to common grid...


g:\ESAproject\FINAL_REPORT\CODE_FINAL\MARINA_EG_ES\pyeoaf\Lib\site-packages\scipy\interpolate\_interpolate.py:712: RuntimeWarning: invalid value encountered in subtract
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
g:\ESAproject\FINAL_REPORT\CODE_FINAL\MARINA_EG_ES\pyeoaf\Lib\site-packages\scipy\interpolate\_interpolate.py:715: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


[WARNING] Regridding mask...
[INFO] Interpolation complete


In [17]:
# --- TIME HANDLING ---
months = (np.arange(spi.sizes["time"]) % 12) + 1

print("[INFO] Time dimension processed")

[INFO] Time dimension processed


In [18]:
# --- FUNCTION ---
def calculate_drought_types(spi_data, spei_data, ssmi_data, n_timesteps, period_name):
    """Compute drought typologies and dominant map"""
    
    print(f"[INFO] Computing drought types for {period_name}...")
    
    # --- CONDITIONS ---
    ssmi_drought = ssmi_data <= drought_threshold
    spi_drought = spi_data <= drought_threshold
    spi_wet = spi_data >= wet_threshold
    spei_drought = spei_data <= drought_threshold
    spei_wet = spei_data >= wet_threshold
    
    # --- DROUGHT TYPES ---
    type1 = ssmi_drought & spi_drought & spei_drought
    type2 = ssmi_drought & spi_drought & spei_wet
    type3 = ssmi_drought & spi_wet & spei_drought
    type4 = ssmi_drought & spi_wet & spei_wet
    
    drought_types = {
        1: ("Compound", type1),
        2: ("Hydrological", type2),
        3: ("Evapotranspiration", type3),
        4: ("Soil_Anomaly", type4)
    }
    
    results = {}
    
    # --- FREQUENCIES ---
    for type_id, (name, condition) in drought_types.items():
        frequency = condition.sum(dim="time") / n_timesteps
        freq_masked = frequency.where(valid_region)
        
        freq_masked.name = f"drought_type_{type_id}_{period_name.lower()}"
        freq_masked.attrs = {
            'long_name': f'{name} Drought Frequency - {period_name}',
            'units': 'fraction'
        }
        
        results[type_id] = (name, freq_masked)
    
    # --- DOMINANT TYPE ---
    frequencies = [drought_types[i][1].sum(dim="time") / n_timesteps for i in [1,2,3,4]]
    freq_stack = xr.concat(frequencies, dim='type')
    
    dominant_type = freq_stack.argmax(dim='type') + 1
    dominant_type = dominant_type.where(freq_stack.max(dim='type') > 0)
    dominant_type = dominant_type.where(valid_region)
    
    dominant_type.name = f"dominant_drought_type_{period_name.lower()}"
    
    return results, dominant_type

In [19]:
# --- MONTHLY ANALYSIS ---
print("[INFO] Starting monthly analysis...")

for target_month in target_months:
    
    month_name = month_names[target_month]
    print(f"\n[INFO] Processing {month_name}...")
    
    month_mask = months == target_month
    
    spi_month = spi.isel(time=month_mask)
    spei_month = spei_interp.isel(time=month_mask)
    ssmi_month = ssmi_interp.isel(time=month_mask)
    
    n_timesteps = spi_month.sizes["time"]
    
    results, dominant_type = calculate_drought_types(
        spi_month, spei_month, ssmi_month, n_timesteps, month_name
    )
    
    # --- SAVE ---
    for type_id, (name, freq_masked) in results.items():
        filename = f"Drought_Type_{type_id}_{name}_{month_name}.nc"
        freq_masked.to_netcdf(os.path.join(base_path, filename))
        print(f"[SAVED] {filename}")
    
    dominant_filename = f"Dominant_Drought_Type_{month_name}.nc"
    dominant_type.to_netcdf(os.path.join(base_path, dominant_filename))
    print(f"[SAVED] {dominant_filename}")

[INFO] Starting monthly analysis...

[INFO] Processing February...
[INFO] Computing drought types for February...
[SAVED] Drought_Type_1_Compound_February.nc
[SAVED] Drought_Type_2_Hydrological_February.nc
[SAVED] Drought_Type_3_Evapotranspiration_February.nc
[SAVED] Drought_Type_4_Soil_Anomaly_February.nc
[SAVED] Dominant_Drought_Type_February.nc

[INFO] Processing May...
[INFO] Computing drought types for May...
[SAVED] Drought_Type_1_Compound_May.nc
[SAVED] Drought_Type_2_Hydrological_May.nc
[SAVED] Drought_Type_3_Evapotranspiration_May.nc
[SAVED] Drought_Type_4_Soil_Anomaly_May.nc
[SAVED] Dominant_Drought_Type_May.nc

[INFO] Processing August...
[INFO] Computing drought types for August...
[SAVED] Drought_Type_1_Compound_August.nc
[SAVED] Drought_Type_2_Hydrological_August.nc
[SAVED] Drought_Type_3_Evapotranspiration_August.nc
[SAVED] Drought_Type_4_Soil_Anomaly_August.nc
[SAVED] Dominant_Drought_Type_August.nc

[INFO] Processing November...
[INFO] Computing drought types for Novem

In [20]:
# --- FULL PERIOD ANALYSIS ---
print("\n[INFO] Processing full period...")

target_mask = np.isin(months, target_months)

spi_filtered = spi.isel(time=target_mask)
spei_filtered = spei_interp.isel(time=target_mask)
ssmi_filtered = ssmi_interp.isel(time=target_mask)

n_timesteps = spi_filtered.sizes["time"]

results_complete, dominant_complete = calculate_drought_types(
    spi_filtered, spei_filtered, ssmi_filtered, n_timesteps, "AllPeriod"
)

# --- SAVE ---
for type_id, (name, freq_masked) in results_complete.items():
    filename = f"Drought_Type_{type_id}_{name}_AllPeriod.nc"
    freq_masked.to_netcdf(os.path.join(base_path, filename))
    print(f"[SAVED] {filename}")

dominant_filename = "Dominant_Drought_Type_AllPeriod.nc"
dominant_complete.to_netcdf(os.path.join(base_path, dominant_filename))

print(f"[SAVED] {dominant_filename}")


[INFO] Processing full period...
[INFO] Computing drought types for AllPeriod...
[SAVED] Drought_Type_1_Compound_AllPeriod.nc
[SAVED] Drought_Type_2_Hydrological_AllPeriod.nc
[SAVED] Drought_Type_3_Evapotranspiration_AllPeriod.nc
[SAVED] Drought_Type_4_Soil_Anomaly_AllPeriod.nc
[SAVED] Dominant_Drought_Type_AllPeriod.nc


In [21]:
# --- SUMMARY ---
n_month_files = len(target_months) * 5
n_total = n_month_files + 5

print("\n[SUCCESS] Analysis completed!")
print("Generated files:")
print(f" - Monthly: {n_month_files}")
print(f" - Full period: 5")
print(f" - Total: {n_total} NetCDF files")


[SUCCESS] Analysis completed!
Generated files:
 - Monthly: 20
 - Full period: 5
 - Total: 25 NetCDF files
